# Task 2 — Streaming Application

This notebook implements the end-to-end streaming pipeline for the AWAS traffic monitoring system. It covers Kafka stream ingestion, stream-to-static joins with camera metadata, instantaneous and average speed violation detection, and MongoDB sink integration.

**Pipeline overview:**
1. Ingest camera event streams from Kafka (Producers A, B, C)
2. Enrich each stream with camera metadata (speed limit, position, coordinates)
3. Detect instantaneous violations — where a vehicle's recorded speed exceeds the camera's speed limit
4. Detect average speed violations — by joining entry/exit events across road segments A→B and B→C
5. Persist all violations to MongoDB, merged by car plate and date

In [ ]:
import glob
import os
import shutil
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-streaming-kafka-0-10_2.12:3.3.0,org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0 pyspark-shell'

from pathlib import Path
from pymongo import MongoClient, UpdateOne
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.functions import (
    col, expr, from_json, to_timestamp, abs as spark_abs,
    unix_timestamp, lit, to_date, sin, cos, sqrt, atan2, radians
)
from pyspark.sql.types import *
from datetime import datetime
import time

HOST_IP = "192.168.64.1"
MONGO_URI = "mongodb://mongodb:27017/"
MONGO_DB = "fit3182_a2"

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('FIT3182-A2')
    .config("spark.sql.shuffle.partitions", "5")
    .config("spark.streaming.stopGracefullyOnShutdown", "true")
    .getOrCreate()
)

print("Debug: SparkSession has been created successfully.")

def log_batch(name):

    def logger(batch_df, batch_id):

        now = datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        )

        row_count = batch_df.count()

        print("\n" + "=" * 70)
        print(f"[{now}] {name}")
        print(f"Spark Batch ID: {batch_id}")
        print(f"Rows received: {row_count}")
        print("=" * 70)

        batch_df.show(
            truncate=False
        )

    return logger

## Task 2.1.2 — Stream Ingestion

Each Kafka topic (`camera-events-A/B/C`) maps to one producer. Events are consumed as JSON, parsed against a fixed schema, and watermarked to bound the stateful join window.

### Event Schema

| Field | Type | Description |
|---|---|---|
| `event_id` | String | Unique identifier for the camera event |
| `batch_id` | Integer | Producer batch sequence number |
| `car_plate` | String | Vehicle licence plate |
| `camera_id` | Integer | Camera that recorded the event |
| `timestamp` | String | ISO timestamp of the recording |
| `speed_reading` | Double | Recorded speed in km/h |

### Join Strategy

Each stream is independently watermarked at **10 minutes**, meaning events more than 10 minutes behind the current max event_time are evicted from state. Segment joins use a physical time-ordering constraint rather than `batch_id`, because `batch_id` is a producer-side sequence number and is not synchronised across producers — a higher `batch_id` in Producer B does not guarantee a later `event_time` than Producer A. Using `batch_id` as a join key would incorrectly drop valid pairs.

### Key Parameters

| Parameter | Value | Rationale |
|---|---|---|
| Watermark duration | 10 minutes | Bounds state size; tolerates producer lag up to 10 min |
| Segment join window | 10 minutes | Max expected travel time between adjacent cameras |
| `startingOffsets` | latest | Process only live events; historical replay not required |
| `shuffle.partitions` | 5 | Reduced from default 200 for local single-node execution |

In [ ]:

# Create a JSON schema that matches the payload from producer for easier handling

event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("batch_id", IntegerType()),
    StructField("car_plate", StringType()),
    StructField("camera_id", IntegerType()),
    StructField("timestamp", StringType()),
    StructField("speed_reading", DoubleType())
])

def read_camera_stream(topic, producer):
    return (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", f"{HOST_IP}:9092")
        .option("subscribe", topic)
        .option("startingOffsets", "latest") # start from first batch
        .load()
        # The value from Kafka is in bytes, so we can cast it to a string
        .selectExpr("CAST(value AS STRING) as json_value")
        # Parse the string into the columns using the struct schema we defined
        .select(from_json(col("json_value"), event_schema).alias("data"))
        .select("data.*")
        # Convert timestamp to proper Spark timestamp type
        .withColumn("event_time", to_timestamp(col("timestamp")))
        # Tag each event with its source
        .withColumn("source", lit(producer))
        .withWatermark("event_time", "10 minutes")
    )

camera_stream_a = read_camera_stream("camera-events-A", "1")
camera_stream_b = read_camera_stream("camera-events-B", "2")
camera_stream_c = read_camera_stream("camera-events-C", "3")

print("Debug: Kafka streams have been created for all three cameras.")

## Camera Metadata Enrichment

Each stream is joined with a static batch read of `camera.csv` (stream-to-static join). This enriches every event with the camera's `speed_limit`, `position`, and GPS coordinates (`latitude`, `longitude`), which are needed for violation detection and Haversine distance calculation.

A stream-to-static join is used here rather than a stream-stream join because camera metadata is fixed and does not change during a run — loading it once as a DataFrame avoids unnecessary state management overhead.

In [ ]:
camera_df = (
    spark.read.csv(f"{Path('..')}/data/camera.csv", header=True, inferSchema=True)
    .select("camera_id", "position", "speed_limit", "latitude", "longitude")
)

camera_df.show()
print(f"Debug: Camera loaded: {camera_df.count()} cameras.")

def join_stream_with_camera(stream):
    return stream.join(camera_df, on="camera_id", how="inner")

joined_stream_a = join_stream_with_camera(camera_stream_a)
joined_stream_b = join_stream_with_camera(camera_stream_b)
joined_stream_c = join_stream_with_camera(camera_stream_c)

## Task 2.1.4 — Instantaneous Speed Violation Detection

A vehicle is flagged for an instantaneous violation when its `speed_reading` at the recording camera exceeds that camera's `speed_limit`. This check is applied independently to each of the three streams.

Detected violations are:
- Written to per-camera JSON output files for local inspection
- Written to the MongoDB `violations` collection via the sink defined in Task 2.1.3

Each violation record retains `event_id` for traceability, and `violation_date` (derived from `event_time`) to support the daily merging logic in MongoDB.

In [ ]:
def get_instant_violations(stream):
    return (
        stream
        .filter(col("speed_reading") > col("speed_limit"))
        .withColumn("violation_type", lit("instantaneous"))
        .withColumn("violation_date", to_date(col("event_time")))
        .select(
            "event_id",
            "car_plate",
            "batch_id",
            "violation_date",
            "violation_type",
            "camera_id",
            col("speed_reading").alias("speed_recorded"),
            "speed_limit",
            col("event_time").cast("string").alias("event_time"),
            "source" 
        )
    )

camera_a_instant_violations = get_instant_violations(joined_stream_a)
camera_b_instant_violations = get_instant_violations(joined_stream_b)
camera_c_instant_violations = get_instant_violations(joined_stream_c)

def write_json_per_batch(base_path):
    def _writer(batch_df, batch_id):
        # Append JSON Lines to a single file so batches accumulate.
        file_path = base_path
        if not file_path.endswith(".json"):
            file_path = f"{base_path}/results.json"
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        rows = batch_df.toJSON().collect()
        if not rows:
            return
        with open(file_path, "a", encoding="utf-8") as f:
            for row in rows:
                f.write(row + "\n")
    return _writer

camera_a_instant_query = (
    camera_a_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/instant_violations_json/camera_a"))
    .start()
)

camera_b_instant_query = (
    camera_b_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/instant_violations_json/camera_b"))
    .start()
)

camera_c_instant_query = (
    camera_c_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/instant_violations_json/camera_c"))
    .start()
)

print("Debug: Instantaneous violations have been extracted and combined.")


## Task 2.1.2 — Average Speed Violation Detection: Segment Joins

Average speed violations are detected by joining entry and exit events for the same vehicle across adjacent road segments. The road layout is strictly A → B → C, so two segment pairs are monitored:

| Segment | Entry Stream | Exit Stream | Join Window |
|---|---|---|---|
| A → B | Producer A (camera 1) | Producer B (camera 2) | 10 minutes |
| B → C | Producer B (camera 2) | Producer C (camera 3) | 10 minutes |

**Join condition:** Events are matched on `car_plate` with a strict time-ordering constraint (`exit.event_time > entry.event_time`, within the 10-minute window). This correctly models the physical requirement that a vehicle must pass the entry camera before the exit camera.

**Dropped pairs:** When no matching exit event arrives for a given entry event within the join window, the entry record is evicted from state by the watermark. These drops are logged by `log_batch_with_drops`.

**Note:** `segment_ab` and `segment_bc` are defined as DataFrames here and passed to `compute_avg_speed` before any `writeStream` is attached. The debug logging queries use separate variables (`debug_query_ab`, `debug_query_bc`) to avoid overwriting the DataFrames.

In [ ]:
# A→B segment join (camera 1 to camera 2)
segment_ab = (
    joined_stream_a.alias("entry")
    .join(
        joined_stream_b.alias("exit"),
        expr("""
            entry.car_plate = exit.car_plate
            AND exit.event_time > entry.event_time
            AND exit.event_time <= entry.event_time + interval 10 minutes
        """),
        "inner"
    )
    .select(
        col("entry.car_plate").alias("car_plate"),
        col("entry.camera_id").alias("start_camera_id"),
        col("exit.camera_id").alias("end_camera_id"),
        col("entry.batch_id").alias("entry_batch_id"),
        col("exit.batch_id").alias("exit_batch_id"),
        col("entry.event_time").alias("entry_time"),
        col("exit.event_time").alias("exit_time"),
        col("entry.position").alias("entry_position"),
        col("exit.position").alias("exit_position"),
        col("exit.speed_limit").alias("speed_limit"),
        col("exit.source").alias("source"),
        col("entry.latitude").alias("entry_latitude"),
        col("entry.longitude").alias("entry_longitude"),
        col("exit.latitude").alias("exit_latitude"),
        col("exit.longitude").alias("exit_longitude")
    )
)

# B→C segment join (camera 2 to camera 3)
segment_bc = (
    joined_stream_b.alias("entry")
    .join(
        joined_stream_c.alias("exit"),
        expr("""
            entry.car_plate = exit.car_plate
            AND exit.event_time > entry.event_time
            AND exit.event_time <= entry.event_time + interval 10 minutes
        """),
        "inner"
    )
    .select(
        col("entry.car_plate").alias("car_plate"),
        col("entry.camera_id").alias("start_camera_id"),
        col("exit.camera_id").alias("end_camera_id"),
        col("entry.batch_id").alias("entry_batch_id"),
        col("exit.batch_id").alias("exit_batch_id"),
        col("entry.event_time").alias("entry_time"),
        col("exit.event_time").alias("exit_time"),
        col("entry.position").alias("entry_position"),
        col("exit.position").alias("exit_position"),
        col("exit.speed_limit").alias("speed_limit"),
        col("exit.source").alias("source"),
        col("entry.latitude").alias("entry_latitude"),
        col("entry.longitude").alias("entry_longitude"),
        col("exit.latitude").alias("exit_latitude"),
        col("exit.longitude").alias("exit_longitude")
    )
)

def log_batch_with_drops(name):
    """Log batch details and flag empty batches as dropped pairs."""
    def logger(batch_df, batch_id):
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        row_count = batch_df.count()
        if row_count == 0:
            print(f"[{now}] [{name}] Batch {batch_id}: NO matching pairs — records dropped/expired by watermark.")
        else:
            print(f"[{now}] [{name}] Batch {batch_id}: {row_count} matched pair(s).")
            batch_df.show(truncate=False)
    return logger

# Debug logging queries — use separate variables so segment_ab/bc DataFrames are not overwritten
debug_query_ab = (
    segment_ab
    .writeStream
    .outputMode("append")
    .foreachBatch(log_batch_with_drops("Segment A→B"))
    .start()
)

debug_query_bc = (
    segment_bc
    .writeStream
    .outputMode("append")
    .foreachBatch(log_batch_with_drops("Segment B→C"))
    .start()
)

print("Debug: Segment joins defined for A→B and B→C.")

## Task 2.1.4 — Average Speed Violation Detection: Computation

For each matched entry/exit pair, the average speed across the segment is computed as:

```
average_speed (km/h) = distance_km / travel_time_hours
```

**Distance** is calculated using the Haversine formula, which gives the great-circle distance between two GPS coordinates. This is more accurate than using the `position` field (which may represent a road-linear distance) for cameras that are not perfectly aligned.

**Travel time** is derived from the difference between `exit_time` and `entry_time`, cast to Unix epoch seconds and divided by 3600 to convert to hours.

A pair is flagged as a violation only when `average_speed > speed_limit` of the **exit camera**, consistent with the AWAS point-to-point enforcement model.

In [ ]:


def calculate_haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371.0  # Earth radius in kilometers

    lat1_rad = radians(col(lat1))
    lon1_rad = radians(col(lon1))
    lat2_rad = radians(col(lat2))
    lon2_rad = radians(col(lon2))

    delta_lat = lat2_rad - lat1_rad
    delta_lon = lon2_rad - lon1_rad

    a = (
        sin(delta_lat / 2) * sin(delta_lat / 2)
        + cos(lat1_rad) * cos(lat2_rad) * sin(delta_lon / 2) * sin(delta_lon / 2)
    )
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance_km = R * c
    return distance_km

def compute_avg_speed(joined_segments):
    return (
        joined_segments
        .withColumn(
            "distance_km",
            calculate_haversine_distance(
                "entry_latitude", "entry_longitude",
                "exit_latitude", "exit_longitude"
            )
        )
        .withColumn(
            "travel_time_hours",
            (col("exit_time").cast("double") - col("entry_time").cast("double")) / 3600
                    )
        .withColumn(
            "average_speed",
            col("distance_km") / col("travel_time_hours")
        )
        .filter(col("average_speed") > col("speed_limit"))
        .withColumn("violation_type", lit("average"))
        .withColumn("violation_date", to_date(col("exit_time")))
        .select(
            "car_plate",
            "violation_date",
            "violation_type",
            "end_camera_id",
            "average_speed",
            "speed_limit",
            col("exit_time").cast("string").alias("event_time"),
            "start_camera_id",
            "distance_km",
            "source",
            "entry_time",
            "exit_time",
            "entry_batch_id",
            "exit_batch_id"
    )
)


average_violations_ab = compute_avg_speed(segment_ab)
average_violations_bc = compute_avg_speed(segment_bc)

ab_avg_violations_query = (
    average_violations_ab
    .writeStream
    .outputMode("append")
    .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/avg_violations/camera_ab"))
    .start()
)

bc_avg_violations_query = (
    average_violations_bc
    .writeStream
    .outputMode("append")
    .foreachBatch(write_json_per_batch(f"{Path('..')}/outputs/avg_violations/camera_bc"))
    .start()
)

# avg_vio_ab_query = (
#     average_violations_ab
#     .writeStream
#     .outputMode("append")
#     .foreachBatch(log_batch("Average Violations A→B"))
#     .start()
# )

# avg_vio_bc_query = (
#     average_violations_bc
#     .writeStream
#     .outputMode("append")
#     .foreachBatch(log_batch("Average Violations B→C"))
#     .start()
# )


print("Average speed violation detection logic defined.")

## Task 2.1.3 — MongoDB Sink

Violations are persisted to the `violations` collection in MongoDB using `foreachBatch` with pymongo `bulk_write` and `UpdateOne` upserts.

### Daily Merging (Task 2.1.4)

Multiple violations for the same vehicle on the same day are merged into a single document. The upsert match key is `(car_plate, date)` — one document per car per day. Each new violation is appended to a `violations` array within that document via `$push`, so all events for the day accumulate in one record.

### Retry Handling

Write failures are retried up to **3 times** with a **2-second delay** between attempts. If all retries are exhausted, the batch is logged as dropped rather than crashing the stream.

### Bulk Writes

All operations within a micro-batch are collected into a single `bulk_write` call with `ordered=False`, which maximises write throughput by allowing MongoDB to execute operations in parallel and not halting on a single failure.

### Indexes

The `violations` collection uses a compound index on `(car_plate, date)` which directly matches the upsert filter key, ensuring O(log n) lookups rather than full collection scans on every write. A secondary index on `date` alone supports time-range queries used in visualisation. See `mongo_setup.py` for index creation.

In [ ]:

def mongo_sink(name):
    def write_violations_to_mongo(batch_df, batch_id):
        rows = batch_df.collect()
        
        if not rows:
            print(f"Batch {batch_id} is empty, skipping MongoDB write. resolving: {name}")
            return

        operations = []
        for row in rows:
            doc = row.asDict()

            # Build the sub-document to push into the violations array
            if doc["violation_type"] == "instantaneous":
                violation_entry = {
                    "type":      "instant",
                    "camera_id": doc["camera_id"],
                    "speed":     doc["speed_recorded"],
                }
            else:
                violation_entry = {
                    "type":         "average",
                    "start_camera": doc["start_camera_id"],
                    "end_camera":   doc["end_camera_id"],
                    "avg_speed":    doc["average_speed"],
                }

            operations.append(
                UpdateOne(
                    # Match key — one document per car per day
                    {
                        "car_plate": doc["car_plate"],
                        "date": datetime.combine(doc["violation_date"], datetime.min.time()),
                    },
                    {
                        # $push always runs — appends to violations array on both
                        # insert and update, so same-day violations accumulate
                        "$push": {"violations": violation_entry},
                    },
                    upsert=True
                )
            )
        
        MAX_RETRIES = 3
        RETRY_DELAY = 2  # seconds
        
        client = MongoClient(MONGO_URI)
        for attempt in range(1, MAX_RETRIES + 1): # HD Requirement
            try:
                collection = client[MONGO_DB]["violations"]
                result = collection.bulk_write(operations, ordered=False)
                print(
                    f"[Batch {batch_id}] [{name}]: {len(operations)} op(s) — "
                    f"upserted: {result.upserted_count}, modified: {result.modified_count}"
                )
                break  # success, exit retry loop
            except Exception as exc:
                print(f"[Batch {batch_id}] [{name}] Attempt {attempt}/{MAX_RETRIES} failed: {exc}")
                if attempt < MAX_RETRIES:
                    time.sleep(RETRY_DELAY)
                else:
                    print(f"[Batch {batch_id}] [{name}] All retries exhausted, batch dropped.")
            finally:
                client.close()
    return write_violations_to_mongo

print("Debug: MongoDB sink function defined.")


## Starting All Streaming Queries

Each violation type and camera combination is wired to a separate `writeStream` query targeting the MongoDB sink. Running them as independent queries allows Spark to manage their trigger schedules and checkpoints separately.

| Query | Source | Violation Type |
|---|---|---|
| `camera_a_query_mongo` | Stream A | Instantaneous |
| `camera_b_query_mongo` | Stream B | Instantaneous |
| `camera_c_query_mongo` | Stream C | Instantaneous |
| `ab_avg_query_mongo` | Segment A→B | Average speed |
| `bc_avg_query_mongo` | Segment B→C | Average speed |

In [ ]:
camera_a_query_mongo = (
    camera_a_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(mongo_sink("camera_a_instant"))
    .start()
)

camera_b_query_mongo = (
    camera_b_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(mongo_sink("camera_b_instant"))
    .start()
)

camera_c_query_mongo = (
    camera_c_instant_violations
    .writeStream
    .outputMode("append")
    .foreachBatch(mongo_sink("camera_c_instant"))
    .start()
)

ab_avg_query_mongo = (
    average_violations_ab
    .writeStream
    .outputMode("append")
    .foreachBatch(mongo_sink("avg_ab"))
    .start()
)

bc_avg_query_mongo = (
    average_violations_bc
    .writeStream
    .outputMode("append")
    .foreachBatch(mongo_sink("avg_bc"))
    .start()
)

print("Debug: MongoDB streaming queries have been started for all violation types.")

## Await Stream Termination

`awaitAnyTermination()` blocks the driver until one of the active streaming queries stops (either naturally or due to an error). This keeps the notebook cell running while all background streaming queries process incoming Kafka events.

In [ ]:
spark.streams.awaitAnyTermination()